# Dormant Kids Incentive - Write Data

This script writes the list of **dormant kids account IDs eligible for the segment offer** into the table `incentive_allocation.fy26_dormant_kids_incentive`.

The dataset must be refreshed on the 1st day of every month to generate a new subset of eligible client IDs.

Eligibility criteria:
- Client status = **Dormant**
- Kid age < **13**

## 1. Create table idempotently

In [1]:
TABLE_NAME = "fy26_dormant_kids_incentive"
STITCHFIX_EMAIL = "sergio.oyola@stitchfix.com"

In [2]:
columns = [
    dict(name="client_id", datatype="int"),
    dict(name="household_primary_client_id", datatype="int"),
    dict(name="as_of", datatype="string", is_partition=True),
]

In [3]:
from bumblebee import BumblebeeClient
from bumblebee.bumblebee_client import BumblebeeError

bbc = BumblebeeClient()

table = f"incentive_allocation.{TABLE_NAME}"

try:
    bbc.create_table(
        table,
        columns=columns,
        permanence="permanent",
        owner="growth-algos@stitchfix.com",
    )
    print(f"Table {table} created successfully.")

except BumblebeeError as e:
    if "already exists" in str(e):
        print(f"Table {table} already exists.")
    else:
        raise

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
Bumblebee client error from (https://fabric-store-http.sos.algo-prod.stitchfix.com/v1/incentive_allocation/fy26_dormant_kids_incentive): {'status': 'bad input', 'message': 'Table incentive_allocation.fy26_dormant_kids_incentive already exists.'}


Table incentive_allocation.fy26_dormant_kids_incentive already exists.


## 2. Load in eligible client IDs

In [4]:
from amphibian import get_data_accessor

da = get_data_accessor(engines=["presto"])

def query(sql):
    return da.fetch_sql(sql=sql, error_on_empty=False)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [5]:
from datetime import date

RUN_DATE = None
run_date = RUN_DATE or date.today()

calendar = query(
    f"""--sql
    SELECT
        CAST(MIN(fiscal_date) AS VARCHAR) AS as_of
    FROM curated_historical.cohorting_calendar
    WHERE DATE_TRUNC('month', fiscal_date) = DATE_TRUNC('month', DATE '{run_date}')
"""
)

as_of = calendar.loc[0, "as_of"]
print(f"run date: {run_date}, as_of partition: {as_of}")

run date: 2026-06-11, as_of partition: 2026-06-01


In [6]:
df = query(
    f"""--sql
	SELECT DISTINCT
    	c.client_id,
		c.household_primary_client_id,
		'{as_of}' as as_of
	FROM curated.client c
	INNER JOIN curated.checkout_based_client_state_journal j
		ON c.client_id = j.client_id
	WHERE
		--kids
		c.business_line = 'Kids'
		--dormant
		AND j.client_state_detail = 'Dormant'
		--haven't aged out
		AND c.age < 13
		-- state as of
		AND DATE(c.start_date) <= DATE('{as_of}')
		AND DATE(c.end_date) > DATE('{as_of}')
"""
)

In [7]:
# Confirm length of df
df.shape[0]

400204

## 3. Write partition to data warehouse

In [8]:
partition_check = query(
    f"""--sql
    SELECT 1
    FROM incentive_allocation.{TABLE_NAME}
    WHERE as_of = '{as_of}'
    LIMIT 1
"""
)

partition_exists = not partition_check.empty

if partition_exists:
    print(f"Partition as_of {as_of} already exists. Skipping extract/save.")
else:
    print(f"Partition as_of {as_of} does not exist. Will proceed with extract/save.")

Partition as_of 2026-06-01 already exists. Skipping extract/save.


In [9]:
import magic_carpet

mcc = magic_carpet.MagicCarpetClient()

if not partition_exists:
    keep_cols = ['client_id', 'household_primary_client_id', 'as_of']
    mcc.save(df[keep_cols], f"incentive_allocation.{TABLE_NAME}",)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [10]:
# Confirm new partitions in the data warehouse
query(
    f"""--sql
    SELECT
        as_of,
        count(client_id) as num_clients,
        count(DISTINCT client_id) as num_unique_clients
    FROM incentive_allocation.{TABLE_NAME}
    GROUP BY 1
    ORDER BY 1 DESC
"""
)

,as_of,num_clients,num_unique_clients
0,2026-06-01,412434,412434
1,2026-05-04,388643,388643
2,2026-04-09,417493,417493
3,2025-12-31,302265,302265


## 4. Delete a partition if needed

In [11]:
# bbc.delete_partition(
#     f'incentive_allocation.{TABLE_NAME}',
#     partition = {'as_of': '2026-05-01'}
# )